In [8]:
import numpy as np
import cv2  # Serve per la dilatazione
import os

# --- CONFIGURAZIONE ---
FILE_IMMAGINI = "/home/michele/Scrivania/train_images.npy"
FILE_MASCHERE = "/home/michele/Scrivania/train_masks.npy"
FILE_OUTPUT   = "/home/michele/Scrivania/masks_cleaned.npy"

# PARAMETRI PER IL VERDE
SOGLIA_VERDE_MIN = 0.2
DIFFERENZA_DOMINANTE = 0.05

# PARAMETRI PER RIFINITURA BORDI
# Aumenta questo valore (es. 5 o 7) se il bordino è spesso
DILATION_KERNEL_SIZE = 3
ITERAZIONI_DILATAZIONE = 2 # Quante volte espandere (1 o 2 di solito basta)

def pulisci_npy_con_dilatazione():
    print("--- CARICAMENTO ---")
    if not os.path.exists(FILE_IMMAGINI):
        print("Errore: File immagini non trovato.")
        return

    imgs = np.load(FILE_IMMAGINI)
    masks = np.load(FILE_MASCHERE)
    cleaned_masks = masks.copy()

    print(f"Shape Immagini: {imgs.shape}")

    # --- RILEVAMENTO FORMATO ---
    if imgs.shape[1] == 3: # (N, 3, H, W)
        print("Formato: Channels First")
        R = imgs[:, 0, :, :]
        G = imgs[:, 1, :, :]
        B = imgs[:, 2, :, :]
        axis_to_expand = 1
    else: # (N, H, W, 3)
        print("Formato: Channels Last")
        R = imgs[:, :, :, 0]
        G = imgs[:, :, :, 1]
        B = imgs[:, :, :, 2]
        axis_to_expand = -1

    print("--- RILEVAMENTO COLORE ---")

    # 1. Trova i pixel sicuramente verdi (Core)
    green_mask_bool = (G > (R + DIFFERENZA_DOMINANTE)) & \
                      (G > (B + DIFFERENZA_DOMINANTE)) & \
                      (G > SOGLIA_VERDE_MIN)

    print(f"Pixel verdi base rilevati: {np.sum(green_mask_bool)}")

    print("--- ESPANSIONE MASCHERA (DILATAZIONE) ---")
    # Convertiamo boolean in uint8 per OpenCV (0 e 255)
    # green_mask_bool è (N, H, W)
    mask_uint8 = (green_mask_bool * 255).astype(np.uint8)

    # Creiamo il kernel (la forma del "pennello" che espande)
    kernel = np.ones((DILATION_KERNEL_SIZE, DILATION_KERNEL_SIZE), np.uint8)

    # Purtroppo cv2.dilate non supporta batch 3D diretti, dobbiamo iterare.
    # Ma essendo maschere binarie, è velocissimo (pochi millisecondi totali).

    final_green_mask = np.zeros_like(green_mask_bool)

    for i in range(len(mask_uint8)):
        # Dilatiamo la maschera dell'i-esima immagine
        # Questo espande le zone bianche (il verde rilevato) mangiando il nero attorno
        dilated = cv2.dilate(mask_uint8[i], kernel, iterations=ITERAZIONI_DILATAZIONE)

        # Salviamo nel risultato (convertendo back a bool se vuoi, o usiamo dilated > 0)
        final_green_mask[i] = dilated > 0

    print(f"Pixel verdi dopo espansione: {np.sum(final_green_mask)}")

    # --- APPLICAZIONE ALLA MASCHERA ---
    # Espandiamo le dimensioni per il broadcasting (N, 1, H, W) o (N, H, W, 1)
    green_mask_expanded = np.expand_dims(final_green_mask, axis=axis_to_expand)

    # Cancelliamo
    cleaned_masks[green_mask_expanded] = 0

    print("--- SALVATAGGIO ---")
    np.save(FILE_OUTPUT, cleaned_masks)
    print(f"Salvato in: {FILE_OUTPUT}")

if __name__ == "__main__":
    pulisci_npy_con_dilatazione()

--- CARICAMENTO ---
Shape Immagini: (1252, 224, 224, 3)
Formato: Channels Last
--- RILEVAMENTO COLORE ---
Pixel verdi base rilevati: 183196
--- ESPANSIONE MASCHERA (DILATAZIONE) ---
Pixel verdi dopo espansione: 245897
--- SALVATAGGIO ---
Salvato in: /home/michele/Scrivania/masks_cleaned.npy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import random

# --- CONFIGURAZIONE ---
FILE_IMMAGINI = "/home/michele/Scrivania/train_images.npy"
FILE_MASCHERE_ORIG = "/home/michele/Scrivania/train_masks.npy"
FILE_MASCHERE_PULITE = "/home/michele/Scrivania/masks_cleaned.npy"

NUM_CAMPIONI = 200  # Quante immagini casuali vuoi vedere

def visualizza_confronto():
    # 1. Caricamento
    print("--- CARICAMENTO DATI ---")
    if not os.path.exists(FILE_MASCHERE_PULITE):
        print("Errore: File maschere pulite non trovato. Esegui prima lo script di pulizia.")
        return

    imgs = np.load(FILE_IMMAGINI)
    masks_orig = np.load(FILE_MASCHERE_ORIG)
    masks_clean = np.load(FILE_MASCHERE_PULITE)

    print(f"Totale immagini: {len(imgs)}")

    # 2. Gestione Formato (Channels First -> Last per Matplotlib)
    # Matplotlib vuole (H, W, 3), se abbiamo (3, H, W) dobbiamo girarli
    channels_first = imgs.shape[1] == 3

    # Indici casuali
    indici = random.sample(range(len(imgs)), NUM_CAMPIONI)

    # 3. Visualizzazione
    for idx in indici:
        img = imgs[idx]
        mask_o = masks_orig[idx]
        mask_c = masks_clean[idx]

        # Adattiamo le dimensioni per il plot
        if channels_first:
            # (3, H, W) -> (H, W, 3)
            img_show = np.transpose(img, (1, 2, 0))
            # (1, H, W) -> (H, W) per la maschera (rimuoviamo asse canali)
            mask_o_show = mask_o[0, :, :]
            mask_c_show = mask_c[0, :, :]
        else:
            # (H, W, 3) -> va bene così
            img_show = img
            # (H, W, 1) -> (H, W)
            mask_o_show = mask_o[:, :, 0]
            mask_c_show = mask_c[:, :, 0]

        # Calcoliamo la differenza (Cosa è stato rimosso?)
        diff = mask_o_show - mask_c_show

        # Creiamo il grafico
        plt.figure(figsize=(16, 4))

        # Immagine Originale
        plt.subplot(1, 4, 1)
        plt.imshow(img_show)
        plt.title(f"IMG Originale (idx {idx})")
        plt.axis('off')

        # Maschera Originale
        plt.subplot(1, 4, 2)
        plt.imshow(mask_o_show, cmap='gray', vmin=0, vmax=1)
        plt.title("Maschera Vecchia")
        plt.axis('off')

        # Maschera Pulita
        plt.subplot(1, 4, 3)
        plt.imshow(mask_c_show, cmap='gray', vmin=0, vmax=1)
        plt.title("Maschera Pulita")
        plt.axis('off')

        # Differenza (Cosa abbiamo tolto)
        plt.subplot(1, 4, 4)
        plt.imshow(diff, cmap='hot', vmin=0, vmax=1)
        plt.title("Zona Rimossa (Verde)")
        plt.axis('off')

        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    visualizza_confronto()